# Co-expression analysis - WGCNA

The count data and the outcome metadata for this analysis are stored in the "input_data_for_wgcna.Rdata" file 

This notebook has three analysis sections:
1. Construction of the gene network and identification of modules for each treatment arm individually
2. Comparison of treatment co-expression network with the Standard of Care
3. Comparing between all three treatment co-expression networks


This notebook uses one treatment arm (tozorakimab) as an example. To analyse the other arms, replace instances of "tozorakimab" with treatment arm of interest. 
  

# Installing packages on Colab

In [ ]:
!pip install PyWGCNA watermark pyvis==0.3.1

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions -v -m -p PyWGCNA,anndata,biomart,gseapy,jedi,json5,numpy,pandas,pyvis,reactome2py,reprit,rsrc

In [6]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(dplyr)

# Setup data

#Setting up the PyWGCNA object

In [ ]:
%%R
study <-"tozorakimab"
# or change to "bemcentinib", "zilucoplan", or "soc"
root.name <-paste(study, sep="")
save.files <- TRUE

In [ ]:
import PyWGCNA
geneExp = 'toz_counts_filtered' # wherever the filt normalised count data has been saved after extraction out of the .Rdata file
pyWGCNA_5xFAD = PyWGCNA.WGCNA(name='tozorakimab', #change for other treatment arms
                              species='homo sapiens',
                              geneExpPath=geneExp,
                              outputPath='',
                              save=True)
pyWGCNA_5xFAD.geneExpr.to_df().head(5)

# change tozorakimab to other treatment arms of interst 

# Pre-processing workflow

PyWGCNA allows you to easily preproces the data including removing genes with too many missing values or lowly-expressed genes across samples (by default we suggest to remove genes without that are expressed below 1 TPM) and removing samples with too many missing values. Keep in your mind that these options can be adjusted by changing TPMcutoff and cut




In [ ]:
TPMcutoff=1
pyWGCNA_5xFAD.preprocess()

## Outlier removal

Optional. 

for this analysis, sample 118007 D1 and D5 matching pair removed for bemcentinib arm due to outlier appearance

In [ ]:
# outlier removal as identified from dendrogram clustering
# Access the geneExpr DataFrame
geneExpr_df = pyWGCNA_5xFAD.geneExpr.to_df()

# List ofsamples  to remove 
outlier = ["Sample_137_118007_D5", "Sample_136_118007_D1"]

# Remove the rows with the specified outliers
geneExpr_df = geneExpr_df[~geneExpr_df.index.isin(outlier)]

In [ ]:
#now need to add this amended df to the wgcna python object
pyWGCNA_5xFAD = PyWGCNA.WGCNA(name='bemcentinib',
                              species='homo sapiens',
                              geneExp=geneExpr_df,
                              outputPath='',
                              save=True)

In [ ]:
#recheck dendrogram
TPMcutoff=1
pyWGCNA_5xFAD.preprocess()

# 1. Construction of the gene network and identification of modules

PyWGCNA compresses all the steps of network construction and module detection in one function called `findModules` which performs the following steps:
1. Choosing the soft-thresholding power: analysis of network topology
2. Co-expression similarity and adjacency
3. Topological Overlap Matrix (TOM)
4. Clustering using TOM
5. Merging of modules whose expression profiles are very similar

In [ ]:
pyWGCNA_5xFAD.findModules()

In [ ]:
print(f"Raw sample informations:")
pyWGCNA_5xFAD.datExpr.obs.head(5)

We also can merge two previous steps by calling `runWGCNA()` function.

### Updating sample information and assiging color to them for dowstream analysis

In [ ]:
pyWGCNA_5xFAD.updateSampleInfo(path='toz_outcome_meta', sep=',') #wherever the metadata for toz has been saved after extraction out of the .Rdata file
#change for other treatment arms

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('D29_Sustained_Clinical_Response', {'Y': 'darkviolet',
                                       'N': 'deeppink'})

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('ICU', {'Y': 'gold',
                                       'N': 'green'})

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('Remdesivir', {'Y': 'sienna',
                                       'N': 'khaki'})

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('Tocilizumab', {'Y': 'gray',
                                       'N': 'purple'})

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('Day', {'D1': 'royalblue',
                                       'D5': 'lightsalmon'})

In [ ]:
# add color for metadata
pyWGCNA_5xFAD.setMetadataColor('D29_Death_RespFailure', {'Y': 'lightcoral',
                                       'N': 'darkolivegreen'})

### Updating gene information

espcially adding gene name for doing downstream analysis

In [ ]:
geneList = PyWGCNA.getGeneList(dataset='hsapiens_gene_ensembl',
                               attributes=['ensembl_gene_id',
                                           'external_gene_name',
                                           'gene_biotype'],
                               maps=['gene_id', 'gene_name', 'gene_biotype'])


pyWGCNA_5xFAD.updateGeneInfo(geneList)

**note**: For doing downstream analysis, we keep aside the Gray modules which is the collection of genes that could not be assigned to any other module.

## Relating modules to external information and identifying important genes
PyWGCNA gather some important analysis after identifying modules in `analyseWGCNA()` function including:

1. Quantifying module–trait relationship
2. Gene relationship to trait and modules
3. Gene-ontology analysis

Before you start analysis, need to add any sample or gene information.

For showing module relationship heatmap, PyWGCNA needs user to choose and set colors from [Matplotlib colors](https://matplotlib.org/stable/gallery/color/named_colors.html) for metadata by using `setMetadataColor()` function.

You also can select which data trait in which order you wish to show in module eigengene heatmap

In [ ]:
del pyWGCNA_5xFAD.geneExpr.obs

In [ ]:
pyWGCNA_5xFAD.analyseWGCNA()

Extracting data and information from the pyWGCNA for subsequent plotting/analysis

In [ ]:
#extract corr and pval info for own plotting
import numpy as np

# Replace these with the correct attribute names
module_trait_cor_matrix = pyWGCNA_5xFAD.moduleTraitCor
module_trait_pvalue_matrix = pyWGCNA_5xFAD.moduleTraitPvalue

# Extract row and column names from the object
row_names_cor = module_trait_cor_matrix.index.tolist()
column_names_cor = module_trait_cor_matrix.columns.tolist()

row_names_pvalue = module_trait_pvalue_matrix.index.tolist()
column_names_pvalue = module_trait_pvalue_matrix.columns.tolist()

# Specify file names for the CSV files
module_trait_cor_file_name = 'tozorkaimb_module_trait_cor_matrix.csv' #change for other treatment arms
module_trait_pvalue_file_name = 'tozorakimab_module_trait_pvalue_matrix.csv' #change for other treatment arms

# Concatenate row names with the values and convert values to strings
data_to_save_cor = np.column_stack((row_names_cor, module_trait_cor_matrix.astype(str).values))
data_to_save_pvalue = np.column_stack((row_names_pvalue, module_trait_pvalue_matrix.astype(str).values))

# Save moduleTraitCor matrix to CSV with row and column names
np.savetxt(module_trait_cor_file_name, data_to_save_cor, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names_cor), encoding='utf-8')

# Save moduleTraitPvalue matrix to CSV with row and column names
np.savetxt(module_trait_pvalue_file_name, data_to_save_pvalue, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names_pvalue), encoding='utf-8')

print(f'Data has been written to {module_trait_cor_file_name} and {module_trait_pvalue_file_name}')


In [ ]:
#extract module eigengene expression for each sample

import pandas as pd

# List of all unique modules
modules = pyWGCNA_5xFAD.datExpr.var['moduleColors'].unique()

# Create an empty DataFrame to store results
all_modules_data = pd.DataFrame()

# Iterate through each module and extract eigengene data
for module in modules:
    # Extract eigengene data (ME) for the current module
    ME = pd.DataFrame(pyWGCNA_5xFAD.datME["ME" + module].values, columns=[f'{module}_eigengeneExp'])
    ME['sample_name'] = pyWGCNA_5xFAD.datME.index

    # Add this module's eigengene data to the results DataFrame
    if all_modules_data.empty:
        all_modules_data = ME
    else:
        all_modules_data = pd.merge(all_modules_data, ME, on='sample_name', how='outer')

# Save the resulting DataFrame to a CSV file
all_modules_data.to_csv('tozorakimab_all_modules_eigengene_data.csv', index=False) #change for other treatment arms

print("CSV file with all modules' eigengene data has been saved to the current working directory.")

## Saving and loading your PyWGCNA
You can save or load your PyWGCNA object with the `saveWGCNA()` or `readWGCNA()` functions respectively.

In [ ]:
pyWGCNA_5xFAD.saveWGCNA()

you can also load your PyWGCNA object with `readWGCNA()` function. you can download `5xFAD.p` from [Zendo](https://zenodo.org/record/6672453#.YrDS4LnMJhE).

In [ ]:
import PyWGCNA
pyWGCNA_5xFAD = PyWGCNA.readWGCNA("Tozorakimab.p") #change for other treatment arms

## Finding hub genes for each module

Extract hub genes from each modules based on their connectivity by using `top_n_hub_genes()` function.

It will give you dataframe sorted by connectivity with additional gene information you have in your expression data.

In [ ]:
pyWGCNA_5xFAD.top_n_hub_genes(moduleName="chocolate", n=50) # chocolate module as an example of a single module

In [ ]:
# here extract all hub genes from all modules present
import pandas as pd

# Function to get all unique module names from the moduleColors column
def get_all_module_names():
    module_colors = pyWGCNA_5xFAD.datExpr.var['moduleColors']
    module_names = module_colors.unique()
    return module_names

# Function to get all hub genes for a given module and save them as CSV
def save_all_hub_genes_for_module(module_name):
    # Get all hub genes for the module by setting a very high n
    hub_genes = pyWGCNA_5xFAD.top_n_hub_genes(moduleName=module_name, n=10000)  # Assuming 10000 is larger than the total number of genes

    # Save the DataFrame to a CSV file
    file_name = f"{module_name}_top_hub_genes_tozorakimab.csv" #change for other treatment arms
    hub_genes.to_csv(file_name, index=True)

# Main function to save hub genes for all modules
def save_hub_genes_for_all_modules():
    module_names = get_all_module_names()
    for module_name in module_names:
        save_all_hub_genes_for_module(module_name)

# Call the main function to save hub genes for all modules
save_hub_genes_for_all_modules()


In [ ]:
print(f"Raw expresion data along with information:\n {pyWGCNA_5xFAD.geneExpr}")

In [ ]:
print(f"Processed expresion data along with information:\n {pyWGCNA_5xFAD.datExpr}")

In [ ]:
import pandas as pd
module_information = pyWGCNA_5xFAD.datExpr.var

# Saving the module information to a CSV file
module_information.to_csv('module_information_tozorakimab.csv') #change for other treatment arms

print("Module information saved to 'module_information_tozorakimab.csv'") #change for other treatment arms


# 2. Comparison of treatment co-expression network with the Standard of Care

Using the saved pyWGCNA object for each individual treatment arm, here we want to compare between one of the drug arms and the standard of care arms:

1. Standard of Care vs Tozorakimab
2. Standard of Care vs Bemcentinib
3. Standard of Care vs Zilucoplan


### Load data
These are the.p files from the pyWGCNA output. Load one drug with the soc each time. 

In [ ]:
#Load
pyWGCNA_soc= PyWGCNA.readWGCNA("soc.p")
pyWGCNA_toz = PyWGCNA.readWGCNA("tozorakimab.p")
#pyWGCNA_bem = PyWGCNA.readWGCNA("bemcentinib.p") 
#pyWGCNA_zil = PyWGCNA.readWGCNA("zilucoplan.p")

## Comparing between standard of care and treatment arm pyWGCNA objects

In [ ]:
comparison = PyWGCNA.compareNetworks(PyWGCNAs = [pyWGCNA_soc, pyWGCNA_toz])

### Jaccard similarity matrix

In [ ]:
# extract to save
jaccard_matrix = comparison.jaccard_similarity
row_names = jaccard_matrix.index.tolist()
column_names = jaccard_matrix.columns.tolist()

# State file name for the Jaccard similarity matrix CSV file
jaccard_file_name = 'jaccard_matrix_soc_treatment.csv' #change this to the treatment being compared

# Concatenate row names with the values and convert values to strings
data_to_save = np.column_stack((row_names, jaccard_matrix.astype(str).values))

# Save Jaccard similarity matrix to CSV with row and column names
np.savetxt(jaccard_file_name, data_to_save, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names), encoding='utf-8')

print(f'Data has been written to {jaccard_file_name}')


### P-value matrix

In [ ]:
# save this
p_value_matrix = comparison.P_value

# Extract row and column names from the object
row_names = p_value_matrix.index.tolist()
column_names = p_value_matrix.columns.tolist()

# Specify file name for the P-value matrix CSV file
p_value_file_name = 'p_value_matrix_soc_treatment.csv' #change this to the treatment being compared

# Concatenate row names with the values and convert values to strings
data_to_save = np.column_stack((row_names, p_value_matrix.astype(str).values))

# Save P-value matrix to CSV with row and column names
np.savetxt(p_value_file_name, data_to_save, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names), encoding='utf-8')

print(f'Data has been written to {p_value_file_name}')


## Plotting results
There are two options to plot the results:
1. Display the Jaccard similarity matrix as a graph by using `plot_jaccard_similarity()` function.
2. Display all three matrices at once using `plotHeatmapComparison()` function.

In [ ]:
# Here using standard of care and tozorakimab as examples
# colours for bemcentinib = lightgreen and for zilucoplan = lightcoral
color = {"soc": "lightblue",
         "tozorakimab": "gold"}
comparison.plotJaccardSimilarity(color=color,
                                 cutoff=0.05,
                                 plot_format="pdf",
                                 file_name="jaccard_similarity_soc_toz")

## Saving and loading your comparison object
You can save or load your comparison object with `saveComparison()` or `readComparison()` functions respectively.

In [ ]:
comparison.saveComparison(name="soc_toz_comparison")

In [ ]:
comparison = PyWGCNA.readComparison('soc_toz_comparison.p')

# 3. Comparing between all three treatment co-expression networks

Comparing for module similarity between the drug treatment arms only
1. Tozorakimab vs Bemcentinib vs Zilucoplan

### Load data

In [ ]:
pyWGCNA_bem= PyWGCNA.readWGCNA("bemcentinib.p")
pyWGCNA_toz = PyWGCNA.readWGCNA("tozorakimab.p")
pyWGCNA_zil = PyWGCNA.readWGCNA("zilucoplan.p")

## Comparing between all three treatment PyWGCNA objects


In [ ]:
comparison = PyWGCNA.compareNetworks(PyWGCNAs = [pyWGCNA_bem, pyWGCNA_toz, pyWGCNA_zil])

### Jacard similarity matrix

In [ ]:
# Replace this with the correct attribute name
jaccard_matrix = comparison.jaccard_similarity

# Extract row and column names from the object
row_names = jaccard_matrix.index.tolist()
column_names = jaccard_matrix.columns.tolist()

# Specify file name for the Jaccard similarity matrix CSV file
jaccard_file_name = 'treatment_consensus_jaccard_matrix.csv'

# Concatenate row names with the values and convert values to strings
data_to_save = np.column_stack((row_names, jaccard_matrix.astype(str).values))

# Save Jaccard similarity matrix to CSV with row and column names
np.savetxt(jaccard_file_name, data_to_save, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names), encoding='utf-8')

print(f'Data has been written to {jaccard_file_name}')


### P-value matrix

In [ ]:
# save this
p_value_matrix = comparison.P_value

# Extract row and column names from the object
row_names = p_value_matrix.index.tolist()
column_names = p_value_matrix.columns.tolist()

# Specify file name for the P-value matrix CSV file
p_value_file_name = 'treatment_consensus_p_value_matrix.csv'

# Concatenate row names with the values and convert values to strings
data_to_save = np.column_stack((row_names, p_value_matrix.astype(str).values))

# Save P-value matrix to CSV with row and column names
np.savetxt(p_value_file_name, data_to_save, delimiter=',', comments='', fmt='%s', header='RowNames,' + ','.join(column_names), encoding='utf-8')

print(f'Data has been written to {p_value_file_name}')


## Plotting results
There are two options to plot the results:
1. Display the Jaccard similarity matrix as a graph by using `plot_jaccard_similarity()` function.
2. Display all three matrices at once using `plotHeatmapComparison()` function.

In [ ]:
color = {"Bemcentinib": "lightgreen",
         "Zilucoplan": "lightcoral",
         "Tozorakimab": "gold"}
comparison.plotJaccardSimilarity(color=color,
                                 cutoff=0.05,
                                 plot_format="pdf",
                                 file_name="treatment_consensus_jaccard_similarity_5xFAD_3xTgAD")

## Saving and loading your comparison object
You can save or load your comparison object with `saveComparison()` or `readComparison()` functions respectively.

In [ ]:
comparison.saveComparison(name="treatment_consensus")

In [ ]:
comparison = PyWGCNA.readComparison('treatment_consensus.p')

Visualisation and further exploration of pyWGCNA output is performed in subsequent R notebook: "05_coexpression_analysis_visualisation"
